# EDA: the disciplined first look at your data

_Data Wrangling_

**Before you model, ask a question, look, notice, guess, check — then ask the next question.**

EDA (Exploratory Data Analysis) is the open-ended detective work you do on a dataset
       before you fit any model. The name and the spirit come from the statistician
       John Tukey, who argued that you should look at data with simple summaries and pictures
       to find out what is going on &mdash; not just run a test to confirm what you already believed.

       The thing to copy from Tukey is the attitude, not a checklist. EDA is a loop:
       ask a question &rarr; look (a summary or a plot) &rarr; notice something &rarr; form a hypothesis
       &rarr; check it &rarr; ask the next question. You go around that loop dozens of times in the first
       hour with a new dataset. Each lap teaches you one fact and points to the next question.

---

This notebook walks the EDA loop one step at a time on a real bundled dataset. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — pandas + matplotlib + seaborn

EDA goes from the whole table down to relationships between columns. We build that
pass in four steps: (1) set up the data, (2) the whole-table view + missingness,
(3) univariate distributions, (4) multivariate structure and the target check.

### Step 1 — Load the data (and remember to split first)

We load the bundled wine dataset as a single DataFrame so the example runs as-is.
On a real project you would **split off a test set first** and explore only the
training rows — looking at the test set lets its information leak into your
decisions. The commented-out `train_test_split` shows where that step belongs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

# Use your own data here; load_wine gives a clean bundled example to run as-is.
data = load_wine(as_frame=True)
df = data.frame                       # 13 numeric features + 'target'
target = 'target'

# IMPORTANT: in a real project, split first and explore the TRAIN set only.
# from sklearn.model_selection import train_test_split
# df, _ = train_test_split(df, test_size=0.2, random_state=0, stratify=df[target])

### Step 2 — The whole-table view, then missingness

Start at the top: how big is the table, what is each column's dtype, and what do the
numeric summaries look like. Then ask which columns are too empty to trust — the
percent-missing per column, worst first. These two looks frame every decision that
follows: a column that is 80% empty changes how you read everything after it.

In [ ]:
# 1) Shape and types -- the whole-table view.
print(df.shape)                       # (rows, columns)
df.info()                             # dtypes + non-null counts per column
print(df.describe().T)                # count/mean/std/min/quartiles/max for numerics

# 2) Missingness -- which columns are too empty to trust?
missing = df.isna().mean().mul(100).sort_values(ascending=False)
print(missing[missing > 0])           # percent missing, worst first

### Step 3 — Univariate: each variable on its own

Before any relationship between variables can be trusted, understand each variable
alone — its skew, its outliers, any hidden missing codes. For numeric columns that
means a histogram grid; for categorical columns, the value counts. The wine data is
all numeric, so the categorical loop simply finds nothing to print.

In [ ]:
# 3) Univariate (numeric): a histogram grid of every numeric column.
num_cols = df.select_dtypes('number').columns.drop(target, errors='ignore')
df[num_cols].hist(figsize=(12, 9), bins=30)
plt.suptitle('Univariate: numeric distributions')
plt.tight_layout()
plt.show()

# 3b) Univariate (categorical): value_counts for each non-numeric column.
cat_cols = df.select_dtypes(exclude='number').columns
for c in cat_cols:
    print(c)
    print(df[c].value_counts(dropna=False).head(10), '\n')

### Step 4 — Multivariate structure and the target check

Now that each variable is understood on its own, look at relationships: a correlation
heatmap reveals which numeric features move together. Finally, check an assumption you
will rely on later — is the target balanced? Write the findings down: skewed columns to
transform, class (im)balance, suspicious correlations. Those notes drive the cleaning.

In [ ]:
# 4) Multivariate: correlation heatmap of the numeric features.
corr = df[num_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True)
plt.title('Numeric correlation heatmap')
plt.tight_layout()
plt.show()

# 5) Check an assumption: is the target balanced?
print(df[target].value_counts())
df[target].value_counts().sort_index().plot.bar()
plt.title('Target distribution')
plt.tight_layout()
plt.show()

# Now WRITE DOWN findings: skewed columns to transform, missing columns to fix,
# class (im)balance, suspicious correlations -- these decisions drive cleaning.

## Visualize the data & results

_On a brand-new dataset (load_wine), what does the very first univariate look reveal? Computing the skew of every numeric feature flags which ones are lopsided and will likely need a transform._

### Step 1 — Load the features and pick out the numeric columns

Reload the wine frame and gather every column except the target. These are the
numeric features whose shape we want to measure in the next step.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

df = load_wine(as_frame=True).frame          # 178 rows, 13 numeric features + target
feats = [c for c in df.columns if c != 'target']

### Step 2 — Rank the features by skew

Compute the skew of each numeric feature and sort it most-right-skewed first — this is
the very first univariate pass. As a rule of thumb, `|skew| > 0.5` is noticeably
lopsided and a candidate for a transform (a log or square-root) before modelling.

In [ ]:
# Skew of every numeric feature, most right-skewed first -- the first univariate pass.
skew = df[feats].skew().sort_values(ascending=False)
print(skew.round(2))
# magnesium        1.10   malic_acid       1.04   color_intensity  0.87
# proline          0.77   proanthocyanins  0.52   nonflavanoid..   0.45
# alcalinity_ash   0.21   total_phenols    0.09   flavanoids       0.03
# hue              0.02   alcohol         -0.05   ash             -0.18
# od280/od315     -0.31
# Rule of thumb: |skew| > 0.5 is noticeably lopsided -> candidate for a transform.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You get a fresh dataset and have 15 minutes before a meeting. In what order do you look at things, and why that order?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Start with shape and dtypes (df.shape, df.info()), then missingness per column. — _These frame everything: how big the data is, what each column is, and which columns are too empty to trust._
- Then go univariate: a histogram for each number, value counts for each category, and the target's distribution. — _You must understand each variable alone &mdash; skew, outliers, hidden missing codes &mdash; before any relationship between variables can be trusted._
- Then go bivariate and multivariate: scatters, a correlation heatmap, distributions split by group or over time. — _Relationships only mean something once you know each variable is clean and well-understood._

**Answer:** Shape/types &rarr; missingness &rarr; univariate &rarr; bivariate &rarr; multivariate. The order goes from the whole table down to relationships. Each step's findings (a column is 80% empty, a feature is skewed) change how you read the next step, and relationship findings are only trustworthy once each variable is understood on its own.

</details>

**Problem 2.** A teammate is excited: a feature correlates 0.98 with the target, so they want to use it. What does disciplined EDA tell you to do first?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Be suspicious of a near-perfect correlation &mdash; real features rarely predict the target that well. — _An almost-perfect correlation is a classic leakage signal: the feature may be a copy of, or computed from, the target._
- Look at how the feature is produced and whether it would be available at prediction time. — _If the value is only known after the outcome (or is derived from it), using it leaks the answer and the model will collapse in production._
- Plot the feature against the target and inspect rows. — _EDA's job here is to find the problem, not to confirm the exciting result._

**Answer:** Treat the 0.98 as a red flag, not a win. A correlation that high usually means leakage &mdash; the feature encodes the target or is unavailable at prediction time. EDA's goal of "find problems" says: investigate where the feature comes from and whether it exists at prediction time before trusting it (see skill-leakage).

</details>

**Problem 3.** Why is it a mistake to draw histograms and correlations on the combined train + test data, and what should you do instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that looking at the test set's distributions and target is a form of peeking. — _Decisions you make after seeing the test set (which features to keep, how to bin, what to drop) bake test information into your pipeline._
- Do all EDA on the training set only. — _This keeps the test set a true held-out estimate of generalization._
- Open the test set only for final evaluation. — _Any earlier look risks an optimistic, irreproducible score._

**Answer:** Exploring the test set lets its information leak into your modeling decisions, inflating your reported score &mdash; the peeking pitfall. Do EDA on the training set only and keep the test set sealed until the final evaluation.

</details>